# 02 — Cleaning, Feature Engineering, and Analytical Dataset

## Purpose

This notebook transforms the validated audit-stage county dataset into a shared analytical dataset for exploratory analysis and modeling.

The input dataset was produced by `01_data_audit.ipynb` and contains one observation for each of 3,144 U.S. counties and county equivalents.

This notebook focuses on:

1. converting source-specific missing-value conventions into consistent representations,
2. creating interpretable population-standardized healthcare-resource features,
3. preparing HPSA shortage measures,
4. preserving appropriate SVI indicators for clustering,
5. documenting feature transformations and limitations,
6. defining candidate supervised and unsupervised feature groups, and
7. saving a common analytical dataset for downstream team workflows.

Model-specific imputation, scaling, hyperparameter tuning, and train/test processing are intentionally deferred to the modeling notebooks to reduce data leakage.

---

## 1. Setup and Load Audit Dataset

The validated output from the data-audit stage is loaded from `data/interim/master_county_audit.csv`.

County FIPS identifiers are explicitly loaded as strings so leading zeros are preserved.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"

RESULTS_DIR = PROJECT_ROOT / "results"
CLEANING_RESULTS_DIR = RESULTS_DIR / "cleaning"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEANING_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
# load audit table

audit_path = (
    INTERIM_DIR
    / "master_county_audit.csv"
)

df = pd.read_csv(
    audit_path,
    dtype={
        "fips": "string"
    },
    low_memory=False
)

print("Loaded:", audit_path)
print("Shape:", df.shape)

Loaded: ../data/interim/master_county_audit.csv
Shape: (3144, 41)


---

## 2. Validate Audit-Stage Input

Before modifying any variables, the notebook verifies that the audit-stage structural guarantees still hold.

The input should contain:

- exactly 3,144 county/county-equivalent observations,
- one unique five-character FIPS identifier per row,
- no duplicate county observations, and
- complete routine-checkup outcome data.

In [4]:
print("Rows:", len(df))
print("Columns:", df.shape[1])

print(
    "Unique FIPS:",
    df["fips"].nunique()
)

print(
    "Duplicate FIPS:",
    df["fips"].duplicated().sum()
)

print(
    "Missing target:",
    df[
        "routine_checkup_pct"
    ].isna().sum()
)

Rows: 3144
Columns: 41
Unique FIPS: 3144
Duplicate FIPS: 0
Missing target: 0


In [5]:
# assertions

assert len(df) == 3144

assert (
    df["fips"].nunique()
    == 3144
)

assert (
    df["fips"]
    .duplicated()
    .sum()
    == 0
)

assert (
    df["fips"]
    .str.len()
    .eq(5)
    .all()
)

assert (
    df[
        "routine_checkup_pct"
    ]
    .isna()
    .sum()
    == 0
)

print(
    "Audit-stage input validation passed."
)

Audit-stage input validation passed.


In [6]:
# create a working copy

clean = df.copy()

Now we have:

- df     = untouched imported audit table
- clean  = working analytical table

---

## 3. Clean Source-Specific Missing Values

The source audit identified multiple forms of missingness. These are handled according to their meaning rather than with a single global rule.

At this stage:

- encoded missing values are converted to standard missing values,
- structural missingness is preserved where meaningful,
- true missing predictor values are retained for later model-specific handling, and
- no statistical imputation is performed.

### 3.1 SVI Sentinel Values

> Dealing with -999 so they don't corrupt distributions, correlations, PCA, and clustering.

In [7]:
# use same candidates

svi_feature_cols = [
    "EP_POV150",
    "EP_UNEMP",
    "EP_HBURD",
    "EP_NOHSDP",
    "EP_UNINSUR",
    "EP_AGE65",
    "EP_AGE17",
    "EP_DISABL",
    "EP_SNGPNT",
    "EP_LIMENG",
    "EP_MINRTY",
    "EP_MUNIT",
    "EP_MOBILE",
    "EP_CROWD",
    "EP_NOVEH",
    "EP_GROUPQ"
]

In [8]:
# check again

svi_sentinel_before = (
    clean[
        svi_feature_cols
    ]
    .eq(-999)
    .sum()
    .sort_values(
        ascending = False
    )
)

svi_sentinel_before

EP_POV150     0
EP_UNEMP      0
EP_HBURD      0
EP_NOHSDP     0
EP_UNINSUR    0
EP_AGE65      0
EP_AGE17      0
EP_DISABL     0
EP_SNGPNT     0
EP_LIMENG     0
EP_MINRTY     0
EP_MUNIT      0
EP_MOBILE     0
EP_CROWD      0
EP_NOVEH      0
EP_GROUPQ     0
dtype: int64

In [9]:
# convert only if they exist 

clean[
    svi_feature_cols
] = (
    clean[
        svi_feature_cols
    ]
    .replace(
        -999,
        np.nan
    )
)

In [10]:
# validate

remaining_sentinels = (
    clean[
        svi_feature_cols
    ]
    .eq(-999)
    .sum()
    .sum()
)

print(
    "Remaining -999 values:",
    remaining_sentinels
)

Remaining -999 values: 0


### SVI Missing-Value Decision

SVI sentinel values of `-999`, where present, are converted to standard missing values (`NaN`).

The raw source files remain unchanged. This conversion is applied only to the analytical working dataset so unavailable values cannot be misinterpreted as valid percentages during EDA, scaling, PCA, or clustering.

### 3.2 AHRF Missingness

In [11]:
# define the source fields:

ahrf_raw_cols = [
    "popn_est_22",
    "phys_nf_prim_care_pc_exc_rsdt_22",
    "np_npi_22",
    "pa_npi_22",
    "hosp_22",
    "hosp_beds_22",
    "fedly_qualfd_hlth_ctr_22",
    "rural_hlth_clincs_22",
    "rural_urban_contnm_23",
    "popn_densty_per_squr_mi_20",
    "pers_noins_lt65_pct_21"
]

In [12]:
# verify

missing_ahrf_columns = [
    col
    for col in ahrf_raw_cols
    if col not in clean.columns
]

missing_ahrf_columns

[]

In [13]:
ahrf_missing_before = pd.DataFrame({
    "missing_n":
        clean[
            ahrf_raw_cols
        ].isna().sum(),

    "missing_pct":
        (
            clean[
                ahrf_raw_cols
            ]
            .isna()
            .mean()
            * 100
        ).round(2)
})

ahrf_missing_before.sort_values(
    "missing_n",
    ascending=False
)

,missing_n,missing_pct
fedly_qualfd_hlth_ctr_22,13,0.41
rural_hlth_clincs_22,13,0.41
pers_noins_lt65_pct_21,10,0.32
hosp_beds_22,9,0.29
np_npi_22,9,0.29
phys_nf_prim_care_pc_exc_rsdt_22,9,0.29
pa_npi_22,9,0.29
popn_densty_per_squr_mi_20,9,0.29
hosp_22,9,0.29
popn_est_22,0,0.00


### AHRF Missing-Value Decision

AHRF predictor missingness is retained during feature engineering rather than immediately imputed.

Several missing values are geographically structured, particularly among current Connecticut planning regions whose 2022 resource data were based on an earlier county geography.

Imputing these values before train/test or cross-validation processing could both conceal the source limitation and introduce leakage.

Supervised-learning imputation will therefore be performed within model pipelines. Unsupervised preprocessing will be handled separately when the clustering feature matrix is constructed.

### 3.3 HPSA Structural Missingness

In [14]:
# define 

hpsa_cols = [
    "pc_hpsa_any",
    "pc_hpsa_count",
    "pc_hpsa_max_score",

    "mh_hpsa_any",
    "mh_hpsa_count",
    "mh_hpsa_max_score",

    "dh_hpsa_any",
    "dh_hpsa_count",
    "dh_hpsa_max_score"
]

In [15]:
# validate

[
    col
    for col in hpsa_cols
    if col not in clean.columns
]

[]

In [16]:
# verify structural missingness

for prefix, label in [
    ("pc", "Primary Care"),
    ("mh", "Mental Health"),
    ("dh", "Dental Health")
]:

    unexpected = clean.loc[
        clean[
            f"{prefix}_hpsa_any"
        ] == 1,
        f"{prefix}_hpsa_max_score"
    ].isna().sum()

    print(
        f"{label}: "
        f"{unexpected} score values missing "
        f"despite designation"
    )

Primary Care: 0 score values missing despite designation
Mental Health: 0 score values missing despite designation
Dental Health: 0 score values missing despite designation


---

## 4. Engineer AHRF Healthcare-Resource Features

Several AHRF healthcare-resource variables are reported as raw county counts.

Because county populations vary substantially, direct comparison of raw counts would largely reflect county size. Where substantively appropriate, counts are converted into population-standardized rates.

Raw variables are retained during development for traceability, while engineered rate variables are created for downstream analysis.

### 4.1 Population Denominator

In [17]:
# validating the denominator before dividing anything

clean[
    "popn_est_22"
].describe()

count    3.144000e+03
mean     1.060075e+05
std      3.322878e+05
min      5.100000e+01
25%      1.076625e+04
50%      2.594400e+04
75%      6.891600e+04
max      9.721138e+06
Name: popn_est_22, dtype: float64

In [18]:
# check invalid populations

invalid_population = (
    clean[
        "popn_est_22"
    ].isna()
    |
    (
        clean[
            "popn_est_22"
        ] <= 0
    )
)

print(
    "Invalid population denominators:",
    invalid_population.sum()
)

Invalid population denominators: 0


In [19]:
# assertions

assert invalid_population.sum() == 0

### 4.2 Provider Rates

> construct rates per 100,000 residents

In [20]:
# create a helper

def rate_per_100k(
    count,
    population
):
    return (
        count
        / population
        * 100_000
    )

In [21]:
clean[
    "primary_care_physicians_per_100k"
] = rate_per_100k(
    clean[
        "phys_nf_prim_care_pc_exc_rsdt_22"
    ],
    clean[
        "popn_est_22"
    ]
)

In [22]:
# Nurse practitioners

clean[
    "nurse_practitioners_per_100k"
] = rate_per_100k(
    clean[
        "np_npi_22"
    ],
    clean[
        "popn_est_22"
    ]
)

In [23]:
# Physician assistants:

clean[
    "physician_assistants_per_100k"
] = rate_per_100k(
    clean[
        "pa_npi_22"
    ],
    clean[
        "popn_est_22"
    ]
)

In [24]:
# inspect

provider_rate_cols = [
    "primary_care_physicians_per_100k",
    "nurse_practitioners_per_100k",
    "physician_assistants_per_100k"
]

clean[
    provider_rate_cols
].describe().T

,count,mean,std,min,25%,50%,75%,max
primary_care_physicians_per_100k,3135.0,50.953613,38.658494,0.0,25.695054,44.538470,69.043846,624.619977
nurse_practitioners_per_100k,3135.0,85.934034,61.579419,0.0,47.850826,75.032827,110.068055,947.700174
physician_assistants_per_100k,3135.0,34.790598,44.580069,0.0,9.140350,24.918449,47.329632,1365.319772


### 4.3 Facility Rates

In [25]:
# Hospitals:

clean[
    "hospitals_per_100k"
] = rate_per_100k(
    clean["hosp_22"],
    clean["popn_est_22"]
)

In [26]:
# FQHCs:

clean[
    "fqhcs_per_100k"
] = rate_per_100k(
    clean[
        "fedly_qualfd_hlth_ctr_22"
    ],
    clean[
        "popn_est_22"
    ]
)

In [27]:
# Rural Health Clinics:

clean[
    "rural_health_clinics_per_100k"
] = rate_per_100k(
    clean[
        "rural_hlth_clincs_22"
    ],
    clean[
        "popn_est_22"
    ]
)

### 4.4 Hospital Bed Rate

In [28]:
# hospital beds are usually more interpretable per 1,000 population

clean[
    "hospital_beds_per_1000"
] = (
    clean[
        "hosp_beds_22"
    ]
    / clean[
        "popn_est_22"
    ]
    * 1_000
)

In [29]:
# review all engineered resource features

ahrf_engineered_cols = [
    "primary_care_physicians_per_100k",
    "nurse_practitioners_per_100k",
    "physician_assistants_per_100k",
    "hospitals_per_100k",
    "hospital_beds_per_1000",
    "fqhcs_per_100k",
    "rural_health_clinics_per_100k"
]

clean[
    ahrf_engineered_cols
].describe().T

,count,mean,std,min,25%,50%,75%,max
primary_care_physicians_per_100k,3135.0,50.953613,38.658494,0.0,25.695054,44.538470,69.043846,624.619977
nurse_practitioners_per_100k,3135.0,85.934034,61.579419,0.0,47.850826,75.032827,110.068055,947.700174
physician_assistants_per_100k,3135.0,34.790598,44.580069,0.0,9.140350,24.918449,47.329632,1365.319772
hospitals_per_100k,3135.0,5.605345,9.426890,0.0,0.838840,2.653576,6.154794,103.466115
hospital_beds_per_1000,3135.0,2.821782,5.008714,0.0,0.537591,1.702651,3.329262,134.328358
fqhcs_per_100k,3131.0,7.455644,14.364377,0.0,0.000000,2.734845,8.345855,207.612457
rural_health_clinics_per_100k,3131.0,10.359612,16.426827,0.0,0.000000,3.444475,14.452608,173.611111


### Healthcare-Resource Rate Decision

Provider and facility counts are converted to population-standardized rates to improve comparability across counties of substantially different population sizes.

The initial rate definitions are:

- primary-care physicians per 100,000 residents,
- nurse practitioners per 100,000 residents,
- physician assistants per 100,000 residents,
- hospitals per 100,000 residents,
- Federally Qualified Health Centers per 100,000 residents,
- Rural Health Clinics per 100,000 residents, and
- hospital beds per 1,000 residents.

No outlier clipping or transformations are applied at this stage. Distributional behavior will be evaluated during EDA.

### 4.5 Rurality and Population Density

For now, retain the original RUCC code.
I would also make a simple metro/nonmetro version because it's interpretable.
USDA RUCC traditionally uses:
- 1–3 = metro
- 4–9 = nonmetro

In [30]:
# create

clean[
    "rucc_metro_status"
] = np.where(
    clean[
        "rural_urban_contnm_23"
    ].between(
        1,
        3
    ),
    "Metro",
    np.where(
        clean[
            "rural_urban_contnm_23"
        ].between(
            4,
            9
        ),
        "Nonmetro",
        pd.NA
    )
)

In [31]:
# check

clean[
    "rucc_metro_status"
].value_counts(
    dropna=False
)

rucc_metro_status
Nonmetro    1958
Metro       1186
Name: count, dtype: int64

In [32]:
# create a log version for later evaluation:

clean[
    "log_population_density"
] = np.log1p(
    clean[
        "popn_densty_per_squr_mi_20"
    ]
)

Why create both?

Because density is usually highly right-skewed:

```
rural county      5 people / sq mile
suburban county   800
urban county      20,000+
```

`log1p()` compresses that range.

> Note: We won't decide which representation is better until EDA.

In [33]:
print(
    "Working dataset shape:",
    clean.shape
)

print(
    "Engineered AHRF features:",
    len(
        ahrf_engineered_cols
    )
)

Working dataset shape: (3144, 50)
Engineered AHRF features: 7


In [34]:
# inspect

clean[
    [
        "fips",
        "state_abbr",
        "county_name_places",
        "routine_checkup_pct",
        "popn_est_22",
        "primary_care_physicians_per_100k",
        "nurse_practitioners_per_100k",
        "hospital_beds_per_1000",
        "fqhcs_per_100k",
        "rucc_metro_status",
        "log_population_density"
    ]
].head(10)

,fips,state_abbr,county_name_places,routine_checkup_pct,popn_est_22,primary_care_physicians_per_100k,nurse_practitioners_per_100k,hospital_beds_per_1000,fqhcs_per_100k,rucc_metro_status,log_population_density
0,06045,CA,Mendocino,68.6,89783.0,80.193355,83.534745,1.314280,15.593152,Nonmetro,3.299534
1,02063,AK,Chugach,62.2,6874.0,145.475706,14.547571,0.000000,NaN,Nonmetro,0.587787
2,04007,AZ,Gila,68.8,53922.0,50.072327,53.781388,1.131264,3.709061,Nonmetro,2.501436
3,05087,AR,Madison,74.5,17486.0,22.875443,51.469747,0.000000,5.718861,Metro,3.034953
4,05067,AR,Jackson,76.7,16624.0,30.076997,102.261790,2.045236,12.030799,Nonmetro,3.310543
5,05113,AR,Polk,75.0,19337.0,77.571495,124.114392,3.361431,5.171433,Nonmetro,3.152736
6,05083,AR,Logan,75.9,21253.0,18.820872,84.693926,1.787983,4.705218,Nonmetro,3.427515
7,01023,AL,Choctaw,78.4,12439.0,40.196157,56.274620,2.009808,8.039231,Nonmetro,2.701361
8,04013,AZ,Maricopa,71.3,4551524.0,68.987882,113.280738,2.503996,1.120504,Metro,6.176699
9,02090,AK,Fairbanks North Star,65.7,95356.0,89.139645,92.285750,2.495910,2.097403,Metro,2.639057


---

## 5. Prepare HPSA Features

The audit-stage dataset contains county-level HPSA measures reconstructed from current HRSA extracts.

For each discipline—Primary Care, Mental Health, and Dental Health—the available county-level measures include:

- presence of at least one qualifying designation,
- number of unique qualifying designations, and
- maximum HPSA score among qualifying designations.

Binary and count measures reflect the approximate year-end 2022 designation reconstruction. Maximum HPSA scores require additional caution because the score stored in the current extract may have been updated after 2022 even when the designation itself was active during 2022.

No HPSA score is assigned to counties without a qualifying designation because the absence of a designation is not equivalent to a score of zero.

### 5.1 Primary Care

In [35]:
# validation 

pc_hpsa_cols = [
    "pc_hpsa_any",
    "pc_hpsa_count",
    "pc_hpsa_max_score"
]

clean[pc_hpsa_cols].describe()

,pc_hpsa_any,pc_hpsa_count,pc_hpsa_max_score
count,3144.000000,3144.000000,2696.000000
mean,0.857506,1.288168,14.655045
std,0.349611,1.664409,3.422462
min,0.000000,0.000000,4.000000
25%,1.000000,1.000000,13.000000
50%,1.000000,1.000000,15.000000
75%,1.000000,1.000000,17.000000
max,1.000000,42.000000,23.000000


In [36]:
# verify the logical relationship between presence and count:

pc_inconsistent = clean[
    (
        (clean["pc_hpsa_any"] == 0)
        & (clean["pc_hpsa_count"] != 0)
    )
    |
    (
        (clean["pc_hpsa_any"] == 1)
        & (clean["pc_hpsa_count"] < 1)
    )
]

print(
    "Primary Care presence/count inconsistencies:",
    len(pc_inconsistent)
)

Primary Care presence/count inconsistencies: 0


In [37]:
# check score structure:

print(
    "PC score missing with designation:",
    clean.loc[
        clean["pc_hpsa_any"] == 1,
        "pc_hpsa_max_score"
    ].isna().sum()
)

print(
    "PC score missing without designation:",
    clean.loc[
        clean["pc_hpsa_any"] == 0,
        "pc_hpsa_max_score"
    ].isna().sum()
)

PC score missing with designation: 0
PC score missing without designation: 448


> second value should correspond to counties with no reconstructed qualifying Primary Care HPSA.

### Primary Care HPSA Decision

`pc_hpsa_any` and `pc_hpsa_count` are retained as candidate analytical features.

`pc_hpsa_any` provides the clearest interpretation of whether a county had at least one qualifying reconstructed Primary Care shortage designation at year-end 2022.

`pc_hpsa_count` captures the number of distinct qualifying designations associated with the county but should be interpreted cautiously because designation counts may reflect both shortage complexity and administrative/geographic structure.

`pc_hpsa_max_score` is retained for sensitivity analysis but is not treated as an exact historical 2022 severity measure because the current HRSA extract may contain scores updated after 2022.

### 5.2 Mental Health

In [38]:
# same checks

mh_hpsa_cols = [
    "mh_hpsa_any",
    "mh_hpsa_count",
    "mh_hpsa_max_score"
]

clean[mh_hpsa_cols].describe()

,mh_hpsa_any,mh_hpsa_count,mh_hpsa_max_score
count,3144.000000,3144.000000,2799.000000
mean,0.890267,1.207697,15.862094
std,0.312606,1.239319,2.919872
min,0.000000,0.000000,5.000000
25%,1.000000,1.000000,14.000000
50%,1.000000,1.000000,17.000000
75%,1.000000,1.000000,18.000000
max,1.000000,31.000000,22.000000


In [39]:
# validate

mh_inconsistent = clean[
    (
        (clean["mh_hpsa_any"] == 0)
        & (clean["mh_hpsa_count"] != 0)
    )
    |
    (
        (clean["mh_hpsa_any"] == 1)
        & (clean["mh_hpsa_count"] < 1)
    )
]

print(
    "Mental Health presence/count inconsistencies:",
    len(mh_inconsistent)
)

Mental Health presence/count inconsistencies: 0


In [40]:
# score

print(
    "MH score missing with designation:",
    clean.loc[
        clean["mh_hpsa_any"] == 1,
        "mh_hpsa_max_score"
    ].isna().sum()
)

print(
    "MH score missing without designation:",
    clean.loc[
        clean["mh_hpsa_any"] == 0,
        "mh_hpsa_max_score"
    ].isna().sum()
)

MH score missing with designation: 0
MH score missing without designation: 345


### Mental Health HPSA Decision

Mental Health HPSA presence and count measures are retained primarily for the unsupervised access–vulnerability analysis.

They may also be evaluated descriptively in relation to routine-checkup utilization, but Mental Health shortage status is less directly aligned with the primary supervised outcome than Primary Care shortage status.

Maximum Mental Health HPSA score is retained as a candidate sensitivity measure subject to the same temporal limitations as the Primary Care score.

> That distinction is useful: PC HPSA is more directly aligned with routine checkups than MH/Dental.

### 5.3 Dental Health

In [41]:
# same processs

dh_hpsa_cols = [
    "dh_hpsa_any",
    "dh_hpsa_count",
    "dh_hpsa_max_score"
]

clean[dh_hpsa_cols].describe()

,dh_hpsa_any,dh_hpsa_count,dh_hpsa_max_score
count,3144.000000,3144.000000,2252.000000
mean,0.716285,0.946565,15.197158
std,0.450872,1.227408,3.661272
min,0.000000,0.000000,2.000000
25%,0.000000,0.000000,13.000000
50%,1.000000,1.000000,16.000000
75%,1.000000,1.000000,18.000000
max,1.000000,27.000000,25.000000


In [42]:
# validate

dh_inconsistent = clean[
    (
        (clean["dh_hpsa_any"] == 0)
        & (clean["dh_hpsa_count"] != 0)
    )
    |
    (
        (clean["dh_hpsa_any"] == 1)
        & (clean["dh_hpsa_count"] < 1)
    )
]

print(
    "Dental Health presence/count inconsistencies:",
    len(dh_inconsistent)
)

Dental Health presence/count inconsistencies: 0


In [43]:
# score check

print(
    "Dental score missing with designation:",
    clean.loc[
        clean["dh_hpsa_any"] == 1,
        "dh_hpsa_max_score"
    ].isna().sum()
)

print(
    "Dental score missing without designation:",
    clean.loc[
        clean["dh_hpsa_any"] == 0,
        "dh_hpsa_max_score"
    ].isna().sum()
)

Dental score missing with designation: 0
Dental score missing without designation: 892


### Dental Health HPSA Decision

Dental HPSA presence and count measures are retained primarily as candidate features for the unsupervised analysis.

They provide additional information about the broader healthcare-shortage environment of a county but are not currently planned as core predictors in the primary supervised routine-checkup model.

Maximum Dental HPSA score is retained only as a candidate sensitivity measure because its value may reflect updates made after 2022.

### 5.4 HPSA Score Limitations

HPSA designation status and HPSA score do not have the same historical certainty in this project.

The approximate year-end 2022 designation status was reconstructed using designation and withdrawal dates:

- designation date on or before December 31, 2022, and
- withdrawal date missing or after December 31, 2022.

This allows reasonably transparent reconstruction of whether a designation appears to have been active at the selected historical cutoff.

However, the HPSA score contained in the current HRSA extract may have been updated after 2022. Therefore, variables such as `pc_hpsa_max_score`, `mh_hpsa_max_score`, and `dh_hpsa_max_score` should be interpreted as:

> the maximum score currently recorded for a qualifying designation that appears to have been active at year-end 2022

rather than:

> the county's exact HPSA score in 2022.

For this reason, HPSA presence indicators are favored over score variables in the primary analyses. Score-based features may be included in sensitivity analyses if they improve interpretation without materially changing conclusions.

In [44]:
# create a combined HPSA validation table (easier to read/compare as one summary)

hpsa_feature_summary = pd.DataFrame({
    "discipline": [
        "Primary Care",
        "Mental Health",
        "Dental Health"
    ],

    "counties_with_designation": [
        clean["pc_hpsa_any"].sum(),
        clean["mh_hpsa_any"].sum(),
        clean["dh_hpsa_any"].sum()
    ],

    "mean_designation_count": [
        clean["pc_hpsa_count"].mean(),
        clean["mh_hpsa_count"].mean(),
        clean["dh_hpsa_count"].mean()
    ],

    "max_designation_count": [
        clean["pc_hpsa_count"].max(),
        clean["mh_hpsa_count"].max(),
        clean["dh_hpsa_count"].max()
    ]
})

hpsa_feature_summary[
    "pct_counties_with_designation"
] = (
    hpsa_feature_summary[
        "counties_with_designation"
    ]
    / len(clean)
    * 100
).round(1)

hpsa_feature_summary

,discipline,counties_with_designation,mean_designation_count,max_designation_count,pct_counties_with_designation
0,Primary Care,2696,1.288168,42,85.8
1,Mental Health,2799,1.207697,31,89.0
2,Dental Health,2252,0.946565,27,71.6


In [45]:
# check HPSA count distributions

clean[
    [
        "pc_hpsa_count",
        "mh_hpsa_count",
        "dh_hpsa_count"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
pc_hpsa_count,3144.0,1.288168,1.664409,0.0,1.0,1.0,1.0,42.0
mh_hpsa_count,3144.0,1.207697,1.239319,0.0,1.0,1.0,1.0,31.0
dh_hpsa_count,3144.0,0.946565,1.227408,0.0,0.0,1.0,1.0,27.0


In [46]:
for col in [
    "pc_hpsa_count",
    "mh_hpsa_count",
    "dh_hpsa_count"
]:
    print(f"\n{col}")
    print(
        clean[col]
        .value_counts()
        .sort_index()
        .head(15)
    )


pc_hpsa_count
pc_hpsa_count
0      448
1     1989
2      524
3       84
4       31
5       13
6        8
7       15
8        9
9        5
10       2
11       3
12       3
14       2
15       1
Name: count, dtype: int64

mh_hpsa_count
mh_hpsa_count
0      345
1     2111
2      611
3       35
4        7
5       10
6        7
7        1
8        3
9        3
10       1
11       1
12       2
13       1
14       1
Name: count, dtype: int64

dh_hpsa_count
dh_hpsa_count
0      892
1     1870
2      286
3       37
4       17
5       11
6        6
7        9
8        2
9        5
10       1
11       2
12       1
14       1
15       1
Name: count, dtype: int64


> Simply checking whether counts are extremely skewed.
> Decision: do not create a log-count feature yet unless the distributions make it necessary during EDA.

In [47]:
# define candidate HPSA feature groups

hpsa_primary_supervised_candidates = [
    "pc_hpsa_any",
    "pc_hpsa_count",
    "pc_hpsa_max_score"
]

But note:

These are candidates, not variables we plan to include all at once.

For unsupervised:

```
hpsa_unsupervised_candidates = [
    "pc_hpsa_any",
    "pc_hpsa_count",
    "mh_hpsa_any",
    "mh_hpsa_count",
    "dh_hpsa_any",
    "dh_hpsa_count"
]
```

Deliberately leave the scores out of the primary clustering feature list for now.

Because score has:
1. historical timing uncertainty;
2. structural missingness for non-designated counties;
3. a different meaning from simple shortage presence;
4. additional preprocessing complications for clustering.
It can still be evaluated later.

##### Important redundancy issue

For each discipline:

```
hpsa_any
```
and
```
hpsa_count
```
are related.

For example:
```
count = 0  → any = 0
count >= 1 → any = 1
```

So including both in the same model may be redundant.

We should preserve both now, but later EDA/model selection should determine whether to use:

#### Option A
```
any only
```

#### Option B
```
count only
```

#### Option C
```
any + score
```
or another justified representation.

### HPSA Feature Redundancy Note

For each discipline, designation presence and designation count contain related information because a positive count necessarily implies designation presence.

Both measures are retained in the shared analytical dataset for transparency, but they should not automatically be included together in final models.

EDA and sensitivity analysis will determine whether presence, count, or another representation provides the most useful and interpretable characterization of shortage status.

In [48]:
# validation: making sure count and binary variables contain no missing values:

hpsa_complete_cols = [
    "pc_hpsa_any",
    "pc_hpsa_count",
    "mh_hpsa_any",
    "mh_hpsa_count",
    "dh_hpsa_any",
    "dh_hpsa_count"
]

print(
    clean[
        hpsa_complete_cols
    ]
    .isna()
    .sum()
)

pc_hpsa_any      0
pc_hpsa_count    0
mh_hpsa_any      0
mh_hpsa_count    0
dh_hpsa_any      0
dh_hpsa_count    0
dtype: int64


In [49]:
assert (
    clean[
        hpsa_complete_cols
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

print(
    "HPSA presence/count feature validation passed."
)

HPSA presence/count feature validation passed.


### HPSA Feature Preparation Result

Primary Care, Mental Health, and Dental Health HPSA presence and designation-count features were validated and retained.

The binary and count measures reflect the approximate year-end 2022 designation reconstruction developed during the data-audit stage.

Maximum HPSA scores were preserved but are treated as secondary candidate features because:

- counties without a qualifying designation have structurally missing scores, and
- scores stored in the current HRSA extract may reflect updates made after 2022.

Primary Care HPSA measures are most directly relevant to the supervised routine-checkup analysis. Mental Health and Dental Health HPSA measures are retained primarily for broader county access–vulnerability profiling in the unsupervised analysis.

No HPSA features were statistically imputed, scaled, combined across disciplines, or transformed during this stage.

---

## 6. Prepare SVI Features

The CDC/ATSDR Social Vulnerability Index (SVI) contributes community-level vulnerability characteristics primarily for the unsupervised analysis.

The analytical dataset contains 16 underlying percentage indicators spanning the four SVI themes. These indicators are retained rather than simultaneously including both component variables and composite SVI rankings, which could double-count related information during clustering.

This section:

1. validates the selected SVI indicators,
2. confirms that sentinel values have been removed,
3. checks value ranges and missingness,
4. organizes variables by SVI theme, and
5. defines the candidate SVI feature set for downstream clustering.

No scaling, PCA, or clustering is performed in this notebook.

### 6.1 Validate Percentage Indicators

In [50]:
# we already defined `svi_feature_cols` earlier, so first verify that all 16 are still available:

missing_svi_columns = [
    col
    for col in svi_feature_cols
    if col not in clean.columns
]

print(
    "Missing expected SVI columns:",
    missing_svi_columns
)

Missing expected SVI columns: []


In [51]:
assert len(
    missing_svi_columns
) == 0

print(
    "All candidate SVI indicators found."
)

All candidate SVI indicators found.


In [52]:
# confirm that -999 is gone

remaining_svi_sentinels = (
    clean[
        svi_feature_cols
    ]
    .eq(-999)
    .sum()
    .sum()
)

print(
    "Remaining SVI -999 values:",
    remaining_svi_sentinels
)

Remaining SVI -999 values: 0


In [53]:
assert (
    remaining_svi_sentinels
    == 0
)

In [54]:
# check numeric types

clean[
    svi_feature_cols
].dtypes

EP_POV150     float64
EP_UNEMP      float64
EP_HBURD      float64
EP_NOHSDP     float64
EP_UNINSUR    float64
EP_AGE65      float64
EP_AGE17      float64
EP_DISABL     float64
EP_SNGPNT     float64
EP_LIMENG     float64
EP_MINRTY     float64
EP_MUNIT      float64
EP_MOBILE     float64
EP_CROWD      float64
EP_NOVEH      float64
EP_GROUPQ     float64
dtype: object

In [55]:
# validate

non_numeric_svi = [
    col
    for col in svi_feature_cols
    if not pd.api.types.is_numeric_dtype(
        clean[col]
    )
]

print(
    "Non-numeric SVI features:",
    non_numeric_svi
)

Non-numeric SVI features: []


In [56]:
# check valid percentage ranges
# These EP_ variables represent percentage estimates
# so valid nonmissing values should ordinarily fall between 0 and 100

svi_range_summary = pd.DataFrame({
    "min":
        clean[
            svi_feature_cols
        ].min(),

    "max":
        clean[
            svi_feature_cols
        ].max(),

    "missing_n":
        clean[
            svi_feature_cols
        ].isna().sum()
})

svi_range_summary

,min,max,missing_n
EP_POV150,2.6,66.6,0
EP_UNEMP,0.0,32.0,0
EP_HBURD,0.0,48.5,0
EP_NOHSDP,0.3,66.7,0
EP_UNINSUR,0.0,45.1,0
EP_AGE65,2.9,57.9,0
EP_AGE17,0.0,41.2,0
EP_DISABL,4.5,41.2,0
EP_SNGPNT,0.0,20.3,0
EP_LIMENG,0.0,36.4,0


In [57]:
# explicitly identify invalid values:

svi_out_of_range = {}

for col in svi_feature_cols:

    invalid = clean[
        clean[col].notna()
        &
        (
            (clean[col] < 0)
            |
            (clean[col] > 100)
        )
    ]

    svi_out_of_range[col] = len(
        invalid
    )

pd.Series(
    svi_out_of_range,
    name="out_of_range_n"
)

EP_POV150     0
EP_UNEMP      0
EP_HBURD      0
EP_NOHSDP     0
EP_UNINSUR    0
EP_AGE65      0
EP_AGE17      0
EP_DISABL     0
EP_SNGPNT     0
EP_LIMENG     0
EP_MINRTY     0
EP_MUNIT      0
EP_MOBILE     0
EP_CROWD      0
EP_NOVEH      0
EP_GROUPQ     0
Name: out_of_range_n, dtype: int64

---

## 7. Feature Quality Checks

Before defining the analytical feature groups, engineered and retained variables are evaluated for data-quality problems that could affect exploratory analysis or modeling.

The checks focus on:

1. missingness after feature engineering,
2. infinite or mathematically invalid values,
3. expected value ranges and unusual distributions,
4. consistency among related variables, and
5. redundant raw and engineered representations.

This stage identifies potential issues but does not automatically remove outliers, impute missing values, or eliminate correlated variables. Those decisions are deferred to EDA and model-specific preprocessing where their effects can be evaluated explicitly.

### 7.1 Missingness After Engineering

Missingness is reassessed after feature engineering to determine whether transformations or rate calculations introduced new unavailable values.

Missing values inherited from the source data are distinguished from values created by invalid denominators or mathematical operations.

In [58]:
# create a list of the main engineered features:

engineered_feature_cols = [
    "primary_care_physicians_per_100k",
    "nurse_practitioners_per_100k",
    "physician_assistants_per_100k",
    "hospitals_per_100k",
    "hospital_beds_per_1000",
    "fqhcs_per_100k",
    "rural_health_clinics_per_100k",
    "log_population_density"
]

In [59]:
# checking that they all exist:

missing_engineered_cols = [
    col
    for col in engineered_feature_cols
    if col not in clean.columns
]

print(
    "Missing engineered columns:",
    missing_engineered_cols
)

Missing engineered columns: []


In [60]:
engineered_missingness = pd.DataFrame({
    "missing_n":
        clean[
            engineered_feature_cols
        ].isna().sum(),

    "missing_pct":
        (
            clean[
                engineered_feature_cols
            ]
            .isna()
            .mean()
            * 100
        ).round(2)
})

engineered_missingness.sort_values(
    "missing_n",
    ascending=False
)

,missing_n,missing_pct
rural_health_clinics_per_100k,13,0.41
fqhcs_per_100k,13,0.41
primary_care_physicians_per_100k,9,0.29
nurse_practitioners_per_100k,9,0.29
hospitals_per_100k,9,0.29
physician_assistants_per_100k,9,0.29
hospital_beds_per_1000,9,0.29
log_population_density,9,0.29


In [61]:
# compare raw vs. engineered missingness
# super useful for the AHRF count → rate transformations

raw_to_engineered = {
    "phys_nf_prim_care_pc_exc_rsdt_22":
        "primary_care_physicians_per_100k",

    "np_npi_22":
        "nurse_practitioners_per_100k",

    "pa_npi_22":
        "physician_assistants_per_100k",

    "hosp_22":
        "hospitals_per_100k",

    "hosp_beds_22":
        "hospital_beds_per_1000",

    "fedly_qualfd_hlth_ctr_22":
        "fqhcs_per_100k",

    "rural_hlth_clincs_22":
        "rural_health_clinics_per_100k"
}

In [62]:
missingness_comparison = []

for raw_col, engineered_col in raw_to_engineered.items():

    missingness_comparison.append({
        "raw_variable":
            raw_col,

        "engineered_variable":
            engineered_col,

        "raw_missing_n":
            clean[
                raw_col
            ].isna().sum(),

        "engineered_missing_n":
            clean[
                engineered_col
            ].isna().sum()
    })

missingness_comparison = pd.DataFrame(
    missingness_comparison
)

missingness_comparison

,raw_variable,engineered_variable,raw_missing_n,engineered_missing_n
0,phys_nf_prim_care_pc_exc_rsdt_22,primary_care_physicians_per_100k,9,9
1,np_npi_22,nurse_practitioners_per_100k,9,9
2,pa_npi_22,physician_assistants_per_100k,9,9
3,hosp_22,hospitals_per_100k,9,9
4,hosp_beds_22,hospital_beds_per_1000,9,9
5,fedly_qualfd_hlth_ctr_22,fqhcs_per_100k,13,13
6,rural_hlth_clincs_22,rural_health_clinics_per_100k,13,13


We should see:
```
raw_missing_n == engineered_missing_n
```
because your population denominator was complete and positive.

In [63]:
# validate

missingness_comparison[
    "new_missing_values"
] = (
    missingness_comparison[
        "engineered_missing_n"
    ]
    -
    missingness_comparison[
        "raw_missing_n"
    ]
)

missingness_comparison

,raw_variable,engineered_variable,raw_missing_n,engineered_missing_n,new_missing_values
0,phys_nf_prim_care_pc_exc_rsdt_22,primary_care_physicians_per_100k,9,9,0
1,np_npi_22,nurse_practitioners_per_100k,9,9,0
2,pa_npi_22,physician_assistants_per_100k,9,9,0
3,hosp_22,hospitals_per_100k,9,9,0
4,hosp_beds_22,hospital_beds_per_1000,9,9,0
5,fedly_qualfd_hlth_ctr_22,fqhcs_per_100k,13,13,0
6,rural_hlth_clincs_22,rural_health_clinics_per_100k,13,13,0


In [64]:
assert (
    missingness_comparison[
        "new_missing_values"
    ] == 0
).all()

print(
    "Rate engineering introduced "
    "no additional missing values."
)

Rate engineering introduced no additional missing values.


### 7.2 Infinite and Invalid Values

Rate calculations and logarithmic transformations can produce invalid values when denominators are zero, negative, or otherwise inappropriate.

The engineered features are therefore screened for positive or negative infinity, impossible negative rates, and other range violations before analytical use.

In [65]:
# check all numeric columns for infinity
# should produce an empty table

numeric_cols = clean.select_dtypes(
    include=np.number
).columns

inf_summary = pd.DataFrame({
    "positive_inf_n": [
        np.isposinf(
            clean[col]
        ).sum()
        for col in numeric_cols
    ],

    "negative_inf_n": [
        np.isneginf(
            clean[col]
        ).sum()
        for col in numeric_cols
    ]
},
    index=numeric_cols
)

inf_summary = inf_summary[
    (
        inf_summary[
            "positive_inf_n"
        ] > 0
    )
    |
    (
        inf_summary[
            "negative_inf_n"
        ] > 0
    )
]

inf_summary

,positive_inf_n,negative_inf_n


In [66]:
assert len(inf_summary) == 0

print(
    "No infinite numeric values detected."
)

No infinite numeric values detected.


In [67]:
# check engineered rates for negative values
# All of these should be nonnegative:

rate_cols = [
    "primary_care_physicians_per_100k",
    "nurse_practitioners_per_100k",
    "physician_assistants_per_100k",
    "hospitals_per_100k",
    "hospital_beds_per_1000",
    "fqhcs_per_100k",
    "rural_health_clinics_per_100k"
]

In [68]:
# check

negative_rate_summary = pd.Series({
    col: (
        clean[col] < 0
    ).sum()
    for col in rate_cols
},
    name="negative_values_n"
)

negative_rate_summary

primary_care_physicians_per_100k    0
nurse_practitioners_per_100k        0
physician_assistants_per_100k       0
hospitals_per_100k                  0
hospital_beds_per_1000              0
fqhcs_per_100k                      0
rural_health_clinics_per_100k       0
Name: negative_values_n, dtype: int64

In [69]:
assert (
    negative_rate_summary
    == 0
).all()

print(
    "No negative healthcare-resource "
    "rates detected."
)

No negative healthcare-resource rates detected.


In [70]:
# check percentage features
# our uninsured AHRF measure should fall within 0–100:

uninsured_invalid = clean[
    clean[
        "pers_noins_lt65_pct_21"
    ].notna()
    &
    (
        (
            clean[
                "pers_noins_lt65_pct_21"
            ] < 0
        )
        |
        (
            clean[
                "pers_noins_lt65_pct_21"
            ] > 100
        )
    )
]

print(
    "Invalid uninsured percentage values:",
    len(uninsured_invalid)
)

Invalid uninsured percentage values: 0


In [71]:
# check RUCC
# Rural-Urban Continuum Code should be within its expected 1–9 range

rucc_invalid = clean[
    clean[
        "rural_urban_contnm_23"
    ].notna()
    &
    ~clean[
        "rural_urban_contnm_23"
    ].between(
        1,
        9
    )
]

print(
    "Invalid RUCC values:",
    len(rucc_invalid)
)

Invalid RUCC values: 0


In [72]:
# checking our derived categorical variable too:
# metro and non-metro plus possibly missing if the original RUCC was missing 

clean[
    "rucc_metro_status"
].value_counts(
    dropna=False
)

rucc_metro_status
Nonmetro    1958
Metro       1186
Name: count, dtype: int64

In [73]:
# check HPSA ranges
# binary fields:

hpsa_any_cols = [
    "pc_hpsa_any",
    "mh_hpsa_any",
    "dh_hpsa_any"
]

for col in hpsa_any_cols:

    invalid_values = set(
        clean[col]
        .dropna()
        .unique()
    ) - {0, 1}

    print(
        col,
        invalid_values
    )

pc_hpsa_any set()
mh_hpsa_any set()
dh_hpsa_any set()


In [74]:
# counts should never be negative:

hpsa_count_cols = [
    "pc_hpsa_count",
    "mh_hpsa_count",
    "dh_hpsa_count"
]

for col in hpsa_count_cols:

    print(
        col,
        "negative values:",
        (clean[col] < 0).sum()
    )

pc_hpsa_count negative values: 0
mh_hpsa_count negative values: 0
dh_hpsa_count negative values: 0


In [75]:
# HPSA scores
# Based on the HRSA scoring system documented for these fields
# HPSA scores should fall within the expected range of 0–26 when present
# check

hpsa_score_cols = [
    "pc_hpsa_max_score",
    "mh_hpsa_max_score",
    "dh_hpsa_max_score"
]

hpsa_score_range = pd.DataFrame({
    "min":
        clean[
            hpsa_score_cols
        ].min(),

    "max":
        clean[
            hpsa_score_cols
        ].max(),

    "missing_n":
        clean[
            hpsa_score_cols
        ].isna().sum()
})

hpsa_score_range

,min,max,missing_n
pc_hpsa_max_score,4.0,23.0,448
mh_hpsa_max_score,5.0,22.0,345
dh_hpsa_max_score,2.0,25.0,892


In [76]:
for col in hpsa_score_cols:

    invalid_n = (
        clean[col].notna()
        &
        ~clean[col].between(
            0,
            26
        )
    ).sum()

    print(
        col,
        "outside 0-26:",
        invalid_n
    )

pc_hpsa_max_score outside 0-26: 0
mh_hpsa_max_score outside 0-26: 0
dh_hpsa_max_score outside 0-26: 0


### 7.3 Distribution and Range Checks

Feature distributions are reviewed at a high level to identify extreme values, strong skew, or unusual ranges that may require further investigation during EDA.

Extreme observations are not automatically treated as errors. In particular, population-standardized healthcare-resource rates can be large in sparsely populated counties even when the underlying source values are valid.

In [77]:
# create a descriptive statistics table

quality_numeric_cols = (
    engineered_feature_cols
    +
    [
        "pers_noins_lt65_pct_21",
        "rural_urban_contnm_23",
        "popn_densty_per_squr_mi_20"
    ]
    +
    hpsa_count_cols
    +
    hpsa_score_cols
    +
    svi_feature_cols
)

In [78]:
# avoid duplicates:

quality_numeric_cols = list(
    dict.fromkeys(
        quality_numeric_cols
    )
)

In [79]:
feature_distribution_summary = (
    clean[
        quality_numeric_cols
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

feature_distribution_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
primary_care_physicians_per_100k,3135.0,50.953613,38.658494,0.0,0.000000,0.000000,25.695054,44.538470,69.043846,118.177178,176.828084,624.619977
nurse_practitioners_per_100k,3135.0,85.934034,61.579419,0.0,0.000000,14.245826,47.850826,75.032827,110.068055,191.778358,292.230157,947.700174
physician_assistants_per_100k,3135.0,34.790598,44.580069,0.0,0.000000,0.000000,9.140350,24.918449,47.329632,103.185400,177.917888,1365.319772
hospitals_per_100k,3135.0,5.605345,9.426890,0.0,0.000000,0.000000,0.838840,2.653576,6.154794,21.623168,48.449612,103.466115
hospital_beds_per_1000,3135.0,2.821782,5.008714,0.0,0.000000,0.000000,0.537591,1.702651,3.329262,8.969230,20.756035,134.328358
fqhcs_per_100k,3131.0,7.455644,14.364377,0.0,0.000000,0.000000,0.000000,2.734845,8.345855,31.205731,73.240803,207.612457
rural_health_clinics_per_100k,3131.0,10.359612,16.426827,0.0,0.000000,0.000000,0.000000,3.444475,14.452608,43.226423,75.197639,173.611111
log_population_density,3135.0,3.857973,1.669420,0.0,0.470004,1.098612,2.839078,3.815512,4.769413,6.862862,8.043003,11.222328
pers_noins_lt65_pct_21,3134.0,11.538928,5.264378,2.4,4.100000,5.200000,7.500000,10.400000,14.600000,21.700000,27.500000,46.300000
rural_urban_contnm_23,3144.0,5.249046,2.929656,1.0,1.000000,1.000000,2.000000,6.000000,8.000000,9.000000,9.000000,9.000000


This is a much better quality-control table than only:
```
describe()
```
because the 1st and 99th percentiles make unusually large values easier to see.

In [80]:
# calculate skewness

feature_skewness = (
    clean[
        quality_numeric_cols
    ]
    .skew(
        numeric_only=True
    )
    .sort_values(
        ascending=False
    )
)

feature_skewness

popn_densty_per_squr_mi_20          26.739558
mh_hpsa_count                       11.350829
pc_hpsa_count                       10.314930
physician_assistants_per_100k       10.261804
hospital_beds_per_1000              10.018877
dh_hpsa_count                        8.552104
EP_NOVEH                             7.655245
EP_CROWD                             5.933926
fqhcs_per_100k                       5.056345
EP_LIMENG                            4.361318
hospitals_per_100k                   4.113206
EP_GROUPQ                            3.713035
EP_MUNIT                             3.664468
rural_health_clinics_per_100k        3.009041
nurse_practitioners_per_100k         2.869798
primary_care_physicians_per_100k     2.563656
EP_UNEMP                             1.860154
EP_UNINSUR                           1.506299
EP_NOHSDP                            1.381215
EP_MINRTY                            1.150795
EP_MOBILE                            1.099065
pers_noins_lt65_pct_21            

In [81]:
# identify skewness

highly_skewed = (
    feature_skewness[
        feature_skewness.abs() > 2
    ]
)

highly_skewed

popn_densty_per_squr_mi_20          26.739558
mh_hpsa_count                       11.350829
pc_hpsa_count                       10.314930
physician_assistants_per_100k       10.261804
hospital_beds_per_1000              10.018877
dh_hpsa_count                        8.552104
EP_NOVEH                             7.655245
EP_CROWD                             5.933926
fqhcs_per_100k                       5.056345
EP_LIMENG                            4.361318
hospitals_per_100k                   4.113206
EP_GROUPQ                            3.713035
EP_MUNIT                             3.664468
rural_health_clinics_per_100k        3.009041
nurse_practitioners_per_100k         2.869798
primary_care_physicians_per_100k     2.563656
dtype: float64

> These features deserve closer inspection during EDA.

In [82]:
# checking specifically the engineered rates

clean[
    engineered_feature_cols
].describe(
    percentiles=[
        0.01,
        0.05,
        0.50,
        0.95,
        0.99
    ]
).T

,count,mean,std,min,1%,5%,50%,95%,99%,max
primary_care_physicians_per_100k,3135.0,50.953613,38.658494,0.0,0.000000,0.000000,44.538470,118.177178,176.828084,624.619977
nurse_practitioners_per_100k,3135.0,85.934034,61.579419,0.0,0.000000,14.245826,75.032827,191.778358,292.230157,947.700174
physician_assistants_per_100k,3135.0,34.790598,44.580069,0.0,0.000000,0.000000,24.918449,103.185400,177.917888,1365.319772
hospitals_per_100k,3135.0,5.605345,9.426890,0.0,0.000000,0.000000,2.653576,21.623168,48.449612,103.466115
hospital_beds_per_1000,3135.0,2.821782,5.008714,0.0,0.000000,0.000000,1.702651,8.969230,20.756035,134.328358
fqhcs_per_100k,3131.0,7.455644,14.364377,0.0,0.000000,0.000000,2.734845,31.205731,73.240803,207.612457
rural_health_clinics_per_100k,3131.0,10.359612,16.426827,0.0,0.000000,0.000000,3.444475,43.226423,75.197639,173.611111
log_population_density,3135.0,3.857973,1.669420,0.0,0.470004,1.098612,3.815512,6.862862,8.043003,11.222328


Pay particular attention to:
```
hospitals_per_100k
fqhcs_per_100k
rural_health_clinics_per_100k
```
Small-population counties can generate very high per-capita facility rates.

Those may be valid.

In [83]:
# identify the counties behind extreme values
# checking whether extreme rates are mostly: tiny rural counties
# rather than obviously impossible values

for col in engineered_feature_cols:

    print(f"\nTop 5: {col}")

    display(
        clean[
            [
                "fips",
                "state_abbr",
                "county_name_places",
                "popn_est_22",
                col
            ]
        ]
        .sort_values(
            col,
            ascending=False
        )
        .head(5)
    )


Top 5: primary_care_physicians_per_100k


,fips,state_abbr,county_name_places,popn_est_22,primary_care_physicians_per_100k
2152,42093,PA,Montour,18091.0,624.619977
1983,38001,ND,Adams,2115.0,378.250591
3029,51610,VA,Falls Church,14586.0,335.938571
880,20025,KS,Clark,1933.0,310.398345
201,02100,AK,Haines,2056.0,291.828794



Top 5: nurse_practitioners_per_100k


,fips,state_abbr,county_name_places,popn_est_22,nurse_practitioners_per_100k
2885,51540,VA,Charlottesville,45373.0,947.700174
2152,42093,PA,Montour,18091.0,724.116964
2831,51840,VA,Winchester,27936.0,572.737686
2888,51720,VA,Norton,3609.0,526.461624
1323,27109,MN,Olmsted,164020.0,504.206804



Top 5: physician_assistants_per_100k


...

,fips,state_abbr,county_name_places,popn_est_22,fqhcs_per_100k
2229,41069,OR,Wheeler,1445.0,207.612457
2636,48311,TX,McMullen,576.0,173.611111
2381,46069,SD,Hyde,1184.0,168.918919
1814,35021,NM,Harding,628.0,159.235669
3007,54075,WV,Pocahontas,7819.0,127.893593



Top 5: rural_health_clinics_per_100k


,fips,state_abbr,county_name_places,popn_est_22,rural_health_clinics_per_100k
1724,31075,NE,Grant,576.0,173.611111
1652,31149,NE,Rock,1245.0,160.642570
2622,48443,TX,Terrell,693.0,144.300144
2677,48263,TX,Kent,740.0,135.135135
316,08053,CO,Hinsdale,775.0,129.032258



Top 5: log_population_density


,fips,state_abbr,county_name_places,popn_est_22,log_population_density
1706,36061,NY,New York,1596273.0,11.222328
1787,36047,NY,Kings,2590516.0,10.582510
1660,36005,NY,Bronx,1379946.0,10.460852
2037,36081,NY,Queens,2278029.0,10.004486
141,06075,CA,San Francisco,808437.0,9.832528


In [84]:
# compare density vs. log density

clean[
    [
        "popn_densty_per_squr_mi_20",
        "log_population_density"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
popn_densty_per_squr_mi_20,3135.0,276.504880,1868.164435,0.0,16.100000,44.400000,116.850000,74780.700000
log_population_density,3135.0,3.857973,1.669420,0.0,2.839078,3.815512,4.769413,11.222328


In [85]:
# and
# log-transformed version likely to be much less right-skewed

clean[
    [
        "popn_densty_per_squr_mi_20",
        "log_population_density"
    ]
].skew()

popn_densty_per_squr_mi_20    26.739558
log_population_density         0.340828
dtype: float64

### Outlier and Transformation Decision

No observations are removed or winsorized during feature-quality screening solely because they appear extreme.

Population-standardized healthcare-resource measures can legitimately take large values in low-population counties. Extreme observations will therefore be examined during EDA before deciding whether transformations, robust scaling, sensitivity analyses, or other treatments are warranted.

Both raw population density and `log_population_density` are retained so their distributions and modeling behavior can be compared.

### 7.4 Redundant Source Variables

We know have:
```
raw count
```
and 
```
population-standardized rate
```
for the concept.

You generally do not want both in the same model.

### 7.4 Redundant Raw and Engineered Variables

The shared dataset retains selected raw AHRF variables for provenance and reproducibility, but raw counts and their population-standardized derivatives should not automatically be used together as model features.

For example, hospital count and hospitals per 100,000 population represent the same underlying resource through different scaling choices.

The engineered rate is generally preferred for cross-county comparison, while the raw source variable is retained for traceability.

In [86]:
# create a mapping table:
# table may be part of feature manifest later

raw_engineered_mapping = pd.DataFrame({
    "concept": [
        "Primary care physicians",
        "Nurse practitioners",
        "Physician assistants",
        "Hospitals",
        "Hospital beds",
        "Federally Qualified Health Centers",
        "Rural Health Clinics"
    ],

    "raw_variable": [
        "phys_nf_prim_care_pc_exc_rsdt_22",
        "np_npi_22",
        "pa_npi_22",
        "hosp_22",
        "hosp_beds_22",
        "fedly_qualfd_hlth_ctr_22",
        "rural_hlth_clincs_22"
    ],

    "preferred_analytical_variable": [
        "primary_care_physicians_per_100k",
        "nurse_practitioners_per_100k",
        "physician_assistants_per_100k",
        "hospitals_per_100k",
        "hospital_beds_per_1000",
        "fqhcs_per_100k",
        "rural_health_clinics_per_100k"
    ]
})

raw_engineered_mapping

,concept,raw_variable,preferred_analytical_variable
0,Primary care physicians,phys_nf_prim_care_pc_exc_rsdt_22,primary_care_physicians_per_100k
1,Nurse practitioners,np_npi_22,nurse_practitioners_per_100k
2,Physician assistants,pa_npi_22,physician_assistants_per_100k
3,Hospitals,hosp_22,hospitals_per_100k
4,Hospital beds,hosp_beds_22,hospital_beds_per_1000
5,Federally Qualified Health Centers,fedly_qualfd_hlth_ctr_22,fqhcs_per_100k
6,Rural Health Clinics,rural_hlth_clincs_22,rural_health_clinics_per_100k


In [87]:
# identify raw fields that should generally not enter models

ahrf_raw_count_cols = (
    raw_engineered_mapping[
        "raw_variable"
    ]
    .tolist()
)

In [88]:
# then

ahrf_preferred_rate_cols = (
    raw_engineered_mapping[
        "preferred_analytical_variable"
    ]
    .tolist()
)

In [89]:
# do not drop the raw variables from `clean`

### Raw-Count Retention Decision

Raw AHRF count variables are retained in the shared processed dataset for provenance and auditing, but the corresponding population-standardized rates are preferred as candidate analytical features.

Feature-group definitions in the next section will exclude redundant raw counts from the primary modeling candidate lists.

Check HPSA redundancy too

```
hpsa_any
hpsa_count
```
are inherently related.

In [90]:
# create quick correlation table 

clean[
    [
        "pc_hpsa_any",
        "pc_hpsa_count",
        "mh_hpsa_any",
        "mh_hpsa_count",
        "dh_hpsa_any",
        "dh_hpsa_count"
    ]
].corr()

,pc_hpsa_any,pc_hpsa_count,mh_hpsa_any,mh_hpsa_count,dh_hpsa_any,dh_hpsa_count
pc_hpsa_any,1.000000,0.315544,0.328499,0.154243,0.407515,0.210617
pc_hpsa_count,0.315544,1.000000,0.105434,0.721533,0.171306,0.669289
mh_hpsa_any,0.328499,0.105434,1.000000,0.342178,0.259866,0.130656
mh_hpsa_count,0.154243,0.721533,0.342178,1.000000,0.143071,0.598391
dh_hpsa_any,0.407515,0.171306,0.259866,0.143071,1.000000,0.485433
dh_hpsa_count,0.210617,0.669289,0.130656,0.598391,0.485433,1.000000


### Population Variable Decision

`popn_est_22` is retained as a reference and feature-engineering denominator but is not automatically included as a primary analytical predictor.

Healthcare-resource rates were created specifically to improve comparability across counties of different population sizes. Whether total population adds independent analytical value will be evaluated separately during EDA rather than assumed.

### 7.5 Feature Quality Summary

In [91]:
# audit table

# Recalculate key quality-check totals for the summary

svi_invalid_total = sum(
    (
        clean[col].notna()
        &
        (
            (clean[col] < 0)
            |
            (clean[col] > 100)
        )
    ).sum()
    for col in svi_feature_cols
)

remaining_svi_sentinels = (
    clean[svi_feature_cols]
    .eq(-999)
    .sum()
    .sum()
)

feature_quality_summary = pd.DataFrame({
    "check": [
        "County rows retained",
        "Duplicate FIPS",
        "Missing target values",
        "Infinite numeric values",
        "Negative engineered rates",
        "Out-of-range SVI percentages",
        "Remaining SVI -999 sentinels"
    ],

    "result": [
        len(clean),
        clean["fips"].duplicated().sum(),
        clean["routine_checkup_pct"].isna().sum(),
        len(inf_summary),
        int(negative_rate_summary.sum()),
        int(svi_invalid_total),
        int(remaining_svi_sentinels)
    ]
})

feature_quality_summary

,check,result
0,County rows retained,3144
1,Duplicate FIPS,0
2,Missing target values,0
3,Infinite numeric values,0
4,Negative engineered rates,0
5,Out-of-range SVI percentages,0
6,Remaining SVI -999 sentinels,0


In [92]:
# final assertions

assert len(clean) == 3144

assert (
    clean["fips"]
    .duplicated()
    .sum()
    == 0
)

assert (
    clean[
        "routine_checkup_pct"
    ]
    .isna()
    .sum()
    == 0
)

assert len(
    inf_summary
) == 0

assert (
    negative_rate_summary
    == 0
).all()

assert (
    remaining_svi_sentinels
    == 0
)

assert (
    svi_invalid_total
    == 0
)

print(
    "Core feature-quality checks passed."
)

Core feature-quality checks passed.


### Feature Quality Check Result

The engineered analytical features passed the core structural and validity checks.

The cleaning and feature-engineering process preserved all 3,144 counties, maintained unique county FIPS identifiers, and retained complete routine-checkup outcome data.

Population-standardized healthcare-resource rates did not introduce additional missing values beyond those already present in their source variables. No infinite values or impossible negative healthcare-resource rates were identified.

SVI percentage indicators were screened for encoded missing values and invalid ranges. HPSA presence, count, and score variables were checked for internal consistency and expected ranges.

Several healthcare-resource variables exhibit substantial skew or extreme per-capita values, particularly in low-population counties. These observations are retained because extreme values are not sufficient evidence of data error. Their influence will be evaluated during EDA and model sensitivity analysis.

Raw AHRF counts are retained for provenance, while their population-standardized rate versions are preferred for cross-county analytical use. Related HPSA representations and alternative population-density measures are also retained so final feature choices can be evaluated empirically rather than imposed during cleaning.

**Decision:** No additional observations or variables are removed solely on the basis of these feature-quality checks. Final redundancy, transformation, and feature-selection decisions are deferred to EDA and modeling.

---

## 8. Define Analytical Feature Groups

The cleaned dataset contains more variables than should be included simultaneously in any single analysis.

To support consistent downstream workflows, variables are organized into analytical feature groups according to their intended role:

1. identifiers and outcome variables,
2. primary supervised-learning candidates,
3. supervised sensitivity candidates,
4. primary unsupervised-learning candidates,
5. unsupervised sensitivity candidates, and
6. reference/provenance variables.

These groups represent the planned starting specifications rather than final feature selections. EDA, redundancy assessment, model diagnostics, and sensitivity analyses may support later changes.

Raw source variables are retained in the shared dataset for provenance even when an engineered version is preferred for analysis.

### 8.1 Identifier and Outcome Variables

In [93]:
# These should never accidentally enter a model as predictors

identifier_cols = [
    "fips",
    "state_abbr",
    "county_name_places",
    "places_target_year"
]

target_col = "routine_checkup_pct"

In [94]:
# validate

for col in identifier_cols + [target_col]:
    assert col in clean.columns

print("Identifier variables:", identifier_cols)
print("Outcome variable:", target_col)

Identifier variables: ['fips', 'state_abbr', 'county_name_places', 'places_target_year']
Outcome variable: routine_checkup_pct


### 8.1 Identifier and Outcome Variables

County identifiers are retained for merging, interpretation, geographic analysis, and reporting but are not analytical predictors.

The supervised-learning outcome is:

`routine_checkup_pct`

which represents county-level age-adjusted prevalence of adults receiving a routine checkup within the past year.

### 8.2 Primary Supervised Candidate Features

For the primary supervised model, I recommend keeping the feature specification relatively focused.

We want it to answer:

> How much can healthcare-resource/access characteristics tell us about routine-checkup utilization?

rather than feeding every available socioeconomic variable into the model.

In [95]:
supervised_primary_numeric = [
    # provider availability
    "primary_care_physicians_per_100k",
    "nurse_practitioners_per_100k",
    "physician_assistants_per_100k",

    # healthcare infrastructure
    "hospitals_per_100k",
    "hospital_beds_per_1000",
    "fqhcs_per_100k",
    "rural_health_clinics_per_100k",

    # geographic/access context
    "log_population_density",

    # primary care shortage
    "pc_hpsa_any"
]

In [96]:
# add categorical feature

supervised_primary_categorical = [
    "rucc_metro_status"
]

In [97]:
# combine for documentation

supervised_primary_candidates = (
    supervised_primary_numeric + supervised_primary_categorical)

print(
    "Primary supervised candidates:",
    len(supervised_primary_candidates)
)

Primary supervised candidates: 10


In [98]:
# validate 

supervised_primary_candidates = (
    supervised_primary_numeric + supervised_primary_categorical)

print(
    "Primary supervised candidates:",
    len(supervised_primary_candidates)
)

Primary supervised candidates: 10


In [99]:
# validate 

missing_supervised_primary = [
    col
    for col in supervised_primary_candidates
    if col not in clean.columns
]

print(
    "Missing primary supervised variables:", missing_supervised_primary)

assert len(missing_supervised_primary) == 0

Missing primary supervised variables: []


### Primary Supervised Feature Strategy

The primary supervised feature set emphasizes healthcare resources, infrastructure, geographic access context, and Primary Care shortage status.

Population-standardized resource measures are preferred over their corresponding raw counts so comparisons are less dominated by county population size.

`pc_hpsa_any` is used as the initial Primary Care shortage representation because its interpretation is clearer and its historical reconstruction is more defensible than the current-extract HPSA score.

The primary specification does not include the full SVI feature set. CDC PLACES small-area estimates incorporate demographic and socioeconomic information, so using strongly overlapping SVI inputs as primary predictors could partially reproduce the outcome-estimation process rather than primarily evaluate healthcare-resource characteristics.

#### NOTE: Why use `pc_hpsa_any` and not all three PC HPSA measures?

```
pc_hpsa_any
pc_hpsa_count
pc_hpsa_max_score
```
are alternative representations of related information. And would not be recommended to use them in the starting model.

`pc_hpsa_any` is the cleanest primary representation.

Then we can count/score in senstivity analysis.


### 8.3 Supervised Sensitivity Candidates

Now we will retain reasonable alternatives without automatically putting them into the primary model.

In [100]:
supervised_sensitivity_candidates = [
    # alternative HPSA representations
    "pc_hpsa_count",
    "pc_hpsa_max_score",

    # alternative rurality representation
    "rural_urban_contnm_23",

    # alternative density representation
    "popn_densty_per_squr_mi_20",

    # additional access/context measure
    "pers_noins_lt65_pct_21"
]

In [101]:
# validate 

missing_supervised_sensitivity = [
    col
    for col in supervised_sensitivity_candidates
    if col not in clean.columns
]

print(
    "Missing supervised sensitivity variables:", missing_supervised_sensitivity)

assert len(
    missing_supervised_sensitivity
) == 0

Missing supervised sensitivity variables: []


### Alternative Supervised Features

*Alternate ways to represent variables already related to the main model, such as HPSA count instead of HPSA presence.*

Several variables are retained as alternative or sensitivity specifications rather than included automatically in the primary supervised model.

These include:

- Primary Care HPSA designation count,
- maximum recorded Primary Care HPSA score,
- the full Rural-Urban Continuum Code,
- untransformed population density, and
- percentage uninsured among residents under age 65.

These variables are useful for evaluating whether results are sensitive to alternative representations of shortage, rurality, density, or insurance context.

The uninsured measure is treated as secondary because it refers specifically to the under-65 population and is measured in 2021 rather than matching the 2022 target exactly.

### Optional SVI Features for Supervised Sensitivity Analysis

*where we add an entirely new social-vulnerability feature family.*

The 16 SVI variables are **not included in the primary supervised model**. They are kept separately so we can test whether adding social-vulnerability information changes model performance or conclusions.

#### How SVI is used in the project

| Analysis | Healthcare Resources / HPSA | SVI Variables | Routine Checkup Outcome |
|---|:---:|:---:|:---:|
| **Primary supervised model** | ✓ | — | **Target** |
| **Supervised sensitivity model** | ✓ | ✓ | **Target** |
| **Primary unsupervised clustering** | ✓ | ✓ | Excluded from clustering |

The supervised comparison can therefore be thought of as:

```text
PRIMARY SUPERVISED MODEL
────────────────────────────────────
Healthcare resources
+ Primary Care HPSA
+ rurality / population density
                ↓
      predict routine checkups


SENSITIVITY MODEL
────────────────────────────────────
Healthcare resources
+ Primary Care HPSA
+ rurality / population density
+ 16 SVI indicators
                ↓
      predict routine checkups

        ↓ COMPARE ↓

Does adding social vulnerability
meaningfully change performance
or conclusions?

In [102]:
# keep SVI variables available for the optional
# broader supervised sensitivity model

supervised_svi_sensitivity = (svi_feature_cols.copy())

### 8.4 Primary Unsupervised Candidate Features

The clustering question is:

> What healthcare-access and social-vulnerability profiles emerge among counties?

So here, using AHRF + HPSA + SVI together makes sense.

In [103]:
unsupervised_resource_features = [
    "primary_care_physicians_per_100k",
    "nurse_practitioners_per_100k",
    "physician_assistants_per_100k",
    "hospitals_per_100k",
    "hospital_beds_per_1000",
    "fqhcs_per_100k",
    "rural_health_clinics_per_100k",
    "log_population_density"
]

In [104]:
# for HPSA, use presence rather than presence + count simultaneously:

unsupervised_hpsa_features = [
    "pc_hpsa_any",
    "mh_hpsa_any",
    "dh_hpsa_any"
]

In [105]:
# SVI

unsupervised_svi_features = (svi_feature_cols.copy())

In [106]:
# combine

unsupervised_primary_candidates = (
    unsupervised_resource_features
    + unsupervised_hpsa_features
    + unsupervised_svi_features
)

print(
    "Primary unsupervised candidates:",
    len(unsupervised_primary_candidates)
)

Primary unsupervised candidates: 27


Gives us:

- 8 resource/context features
- 3 HPSA features
- 16 SVI features
  
for 27 initial clustering candidates.

In [107]:
# validate

missing_unsupervised_primary = [
    col
    for col in unsupervised_primary_candidates
    if col not in clean.columns
]

print(
    "Missing primary unsupervised variables:", missing_unsupervised_primary)

assert len(
    missing_unsupervised_primary
) == 0

Missing primary unsupervised variables: []


### Primary Unsupervised Feature Strategy

The initial clustering candidate set combines three conceptual domains:

**Healthcare-resource availability**
- provider supply,
- healthcare facilities,
- hospital capacity,
- and population density.

**Healthcare shortage**
- Primary Care,
- Mental Health,
- and Dental Health HPSA designation presence.

**Social vulnerability**
- the 16 underlying SVI percentage indicators.

HPSA designation-presence measures are used in the initial clustering specification rather than simultaneously including both presence and designation count, which would give closely related shortage information multiple representations.

The final clustering feature set will be refined during unsupervised EDA based on redundancy, correlation structure, missingness, cluster stability, and substantive interpretability.

#### Important: routine-checkup prevalence is NOT in this list

In [108]:
# explicit test

assert (
    target_col
    not in unsupervised_primary_candidates
)

print(
    "PLACES target is excluded "
    "from clustering features."
)

PLACES target is excluded from clustering features.


The PLACES routine-checkup outcome is deliberately excluded from cluster construction.

After clusters are created independently, routine-checkup prevalence may be compared across clusters as an external descriptive outcome.

### 8.5 Unsupervised Sensitivity Candidates

In [109]:
# some alternatives:

unsupervised_sensitivity_candidates = [
    # alternative HPSA representations
    "pc_hpsa_count",
    "mh_hpsa_count",
    "dh_hpsa_count",

    # alternative population-density representation
    "popn_densty_per_squr_mi_20",

    # rurality
    "rural_urban_contnm_23",

    # additional insurance context
    "pers_noins_lt65_pct_21"
]

In [110]:
# retain HPSA scores separately:

unsupervised_hpsa_score_sensitivity = [
    "pc_hpsa_max_score",
    "mh_hpsa_max_score",
    "dh_hpsa_max_score"
]

#### Note: 

May be best to keep score sensitivity separate because of the temporal caveat and structural missingness.

### Unsupervised Sensitivity Features

Alternative clustering specifications may evaluate:

- HPSA designation counts instead of binary presence,
- the original population-density measure instead of its log transformation,
- Rural-Urban Continuum Code,
- uninsured prevalence, and
- HPSA scores as a secondary sensitivity analysis.

HPSA scores are separated from the primary candidate matrix because they contain structural missingness for counties without a qualifying designation and may reflect score updates occurring after 2022.

### 8.6 Reference and Provenance Variables

Explicitly identify variables that should generally not go into models.

In [111]:
# raw AHRF counts:

reference_raw_ahrf = [
    "popn_est_22",
    "phys_nf_prim_care_pc_exc_rsdt_22",
    "np_npi_22",
    "pa_npi_22",
    "hosp_22",
    "hosp_beds_22",
    "fedly_qualfd_hlth_ctr_22",
    "rural_hlth_clincs_22"
]

In [112]:
reference_cols = (identifier_cols + reference_raw_ahrf)

### Reference and Provenance Variables

Several variables are retained in the shared dataset for provenance, validation, interpretation, or feature engineering but are not intended as default model predictors.

These include:

- county identifiers,
- population denominator,
- raw AHRF provider/facility counts whose rate-based versions are preferred, and
- alternative representations retained for sensitivity analysis.

Retaining these fields makes the processed dataset auditable without requiring downstream analysts to return to the raw source files.

### 8.7 Create a Feature Manifest

> a table to describe each feature

In [113]:
# helper

feature_manifest_rows = []

In [114]:
# identifiers:

for col in identifier_cols:
    feature_manifest_rows.append({
        "variable": col,
        "role": "identifier",
        "analysis": "reference",
        "primary": False
    })

In [115]:
# target:

feature_manifest_rows.append({
    "variable": target_col,
    "role": "outcome",
    "analysis": "supervised",
    "primary": True
})

In [116]:
# Supervised primary

for col in supervised_primary_candidates:
    feature_manifest_rows.append({
        "variable": col,
        "role": "predictor",
        "analysis": "supervised",
        "primary": True
    })

In [117]:
# alternative supervised sensitivity features:

for col in supervised_sensitivity_candidates:
    feature_manifest_rows.append({
        "variable": col,
        "role": "predictor",
        "analysis": "supervised_sensitivity",
        "primary": False
    })

In [118]:
# SVI variables retained for the optional broader
# supervised sensitivity model

for col in supervised_svi_sensitivity:
    feature_manifest_rows.append({
        "variable": col,
        "role": "predictor",
        "analysis": "supervised_svi_sensitivity",
        "primary": False
    })

In [119]:
# Unsupervised primary:

for col in unsupervised_primary_candidates:
    feature_manifest_rows.append({
        "variable": col,
        "role": "clustering_feature",
        "analysis": "unsupervised",
        "primary": True
    })

In [120]:
# Unsupervised sensitivity

for col in unsupervised_sensitivity_candidates:
    feature_manifest_rows.append({
        "variable": col,
        "role": "clustering_feature",
        "analysis": "unsupervised_sensitivity",
        "primary": False
    })

In [121]:
# HPSA score variables retained separately because of
# temporal uncertainty and structural missingness
for col in unsupervised_hpsa_score_sensitivity:
    feature_manifest_rows.append({
        "variable": col,
        "role": "clustering_feature",
        "analysis": "unsupervised_hpsa_score_sensitivity",
        "primary": False
    })

In [122]:
# raw AHRF variables retained for provenance and auditing
for col in reference_raw_ahrf:
    feature_manifest_rows.append({
        "variable": col,
        "role": "reference",
        "analysis": "reference",
        "primary": False
    })

In [123]:
# create

feature_manifest = pd.DataFrame(feature_manifest_rows)

In [124]:
# inspect

feature_manifest.head(20)

,variable,role,analysis,primary
0,fips,identifier,reference,False
1,state_abbr,identifier,reference,False
2,county_name_places,identifier,reference,False
3,places_target_year,identifier,reference,False
4,routine_checkup_pct,outcome,supervised,True
5,primary_care_physicians_per_100k,predictor,supervised,True
6,nurse_practitioners_per_100k,predictor,supervised,True
7,physician_assistants_per_100k,predictor,supervised,True
8,hospitals_per_100k,predictor,supervised,True
9,hospital_beds_per_1000,predictor,supervised,True


In [125]:
# check for accidental duplicate variable-analysis pairs
duplicate_manifest_pairs = (
    feature_manifest[
        ["variable", "analysis"]
    ]
    .duplicated()
    .sum()
)

print(
    "Manifest rows:",
    len(feature_manifest)
)

print(
    "Unique variables:",
    feature_manifest["variable"].nunique()
)

print(
    "Duplicate variable-analysis pairs:",
    duplicate_manifest_pairs
)

assert duplicate_manifest_pairs == 0

Manifest rows: 80
Unique variables: 50
Duplicate variable-analysis pairs: 0


```
80 manifest rows = variable × analytical-role entries
50 unique variables = actual distinct columns
0 duplicate variable-analysis pairs = no accidental duplicate entries
```

In [126]:
# variables may legitimately appear in more than one
# analytical group
repeated_variables = (
    feature_manifest[
        feature_manifest["variable"].duplicated(
            keep=False
        )
    ]
    .sort_values(
        ["variable", "analysis"]
    )
)

repeated_variables

,variable,role,analysis,primary
26,EP_AGE17,predictor,supervised_svi_sensitivity,False
53,EP_AGE17,clustering_feature,unsupervised,True
25,EP_AGE65,predictor,supervised_svi_sensitivity,False
52,EP_AGE65,clustering_feature,unsupervised,True
33,EP_CROWD,predictor,supervised_svi_sensitivity,False
60,EP_CROWD,clustering_feature,unsupervised,True
27,EP_DISABL,predictor,supervised_svi_sensitivity,False
54,EP_DISABL,clustering_feature,unsupervised,True
35,EP_GROUPQ,predictor,supervised_svi_sensitivity,False
62,EP_GROUPQ,clustering_feature,unsupervised,True


In [127]:
# final summary for easier inspection

feature_manifest.groupby(
    ["analysis", "primary"]
).size().reset_index(
    name="n_variables"
)

,analysis,primary,n_variables
0,reference,False,12
1,supervised,True,11
2,supervised_sensitivity,False,5
3,supervised_svi_sensitivity,False,16
4,unsupervised,True,27
5,unsupervised_hpsa_score_sensitivity,False,3
6,unsupervised_sensitivity,False,6


### 8.8 Final Feature-Group Validation

In [128]:
# clean overview
feature_group_summary = pd.DataFrame({
    "feature_group": [
        "Identifiers",
        "Primary supervised",
        "Supervised sensitivity",
        "Supervised SVI sensitivity",
        "Primary unsupervised",
        "Unsupervised sensitivity",
        "HPSA score sensitivity"
    ],

    "n_variables": [
        len(identifier_cols),
        len(supervised_primary_candidates),
        len(supervised_sensitivity_candidates),
        len(supervised_svi_sensitivity),
        len(unsupervised_primary_candidates),
        len(unsupervised_sensitivity_candidates),
        len(unsupervised_hpsa_score_sensitivity)
    ]
})

feature_group_summary

,feature_group,n_variables
0,Identifiers,4
1,Primary supervised,10
2,Supervised sensitivity,5
3,Supervised SVI sensitivity,16
4,Primary unsupervised,27
5,Unsupervised sensitivity,6
6,HPSA score sensitivity,3


In [129]:
# verify no raw resource counts accidentally entered primary groups

primary_analysis_features = set(
    supervised_primary_candidates
    + unsupervised_primary_candidates
)

raw_counts_in_primary = (
    primary_analysis_features
    & set(ahrf_raw_count_cols)
)

print(
    "Raw AHRF counts in primary feature groups:",
    raw_counts_in_primary
)

Raw AHRF counts in primary feature groups: set()


In [130]:
# verify target isn't in predictor sets

assert (
    target_col
    not in supervised_primary_candidates
)

assert (
    target_col
    not in unsupervised_primary_candidates
)

print(
    "Outcome excluded from all predictor "
    "and clustering feature groups."
)

Outcome excluded from all predictor and clustering feature groups.


In [131]:
# verify no identifiers entered models 

model_features = set(
    supervised_primary_candidates
    + unsupervised_primary_candidates
)

identifier_leakage = (
    model_features
    & set(identifier_cols)
)

print(
    "Identifiers in model features:",
    identifier_leakage
)

Identifiers in model features: set()


### Analytical Feature Group Result

The cleaned variables were organized into explicit analytical feature groups to provide consistent starting points for the supervised and unsupervised workflows.

The primary supervised specification emphasizes population-standardized healthcare-resource measures, geographic/access context, and Primary Care HPSA designation status. Broader socioeconomic vulnerability measures and alternative shortage representations are retained separately for sensitivity analyses.

The primary unsupervised specification combines healthcare-resource availability, HPSA shortage presence across Primary Care, Mental Health, and Dental Health, and the 16 underlying SVI indicators.

Raw AHRF counts remain available for provenance but are excluded from the primary feature groups where population-standardized versions are available.

The PLACES routine-checkup outcome is excluded from all clustering features and will only be used after clustering for external descriptive comparison.

These feature groups represent starting specifications rather than final selections. EDA, redundancy assessment, cross-validation, clustering stability, and sensitivity analyses may support subsequent refinement.

**Decision:** The processed dataset will preserve all validated source and engineered variables, while the feature manifest will identify the intended role of each variable for downstream team analysis.

#### Ready for handoff

**Sara** (supervised model) gets:
```
target_col

supervised_primary_numeric
supervised_primary_categorical
supervised_sensitivity_candidates
supervised_svi_sensitivity
```

**Gina** (unsupervised model) gets:
```
unsupervised_primary_candidates
unsupervised_sensitivity_candidates
unsupervised_hpsa_score_sensitivity
```

---

## 9. Create Shared Analytical Dataset

The cleaned working dataset is now reduced to the variables documented in the feature manifest.

The resulting shared analytical dataset serves as the common input for supervised and unsupervised EDA and modeling.

Rather than creating separate source datasets for each analysis, one shared county-level dataset is preserved. The accompanying feature manifest identifies which variables belong to:

- the primary supervised specification,
- supervised sensitivity analyses,
- the primary unsupervised specification,
- unsupervised sensitivity analyses,
- identifiers and outcome variables, and
- reference/provenance fields.

This approach keeps both analytical workflows synchronized while allowing each teammate to select the variables appropriate to their analysis.

Model-specific imputation, encoding, scaling, feature selection, and dimensionality reduction remain deferred to the downstream modeling workflows.

### 9.1 Define Shared Dataset Columns

In [132]:
# feature manifest = source of truth

# collect every unique variable represented in the feature manifest
shared_dataset_cols = (
    feature_manifest["variable"]
    .drop_duplicates()
    .tolist()
)

print(
    "Variables in shared analytical dataset:",
    len(shared_dataset_cols)
)

Variables in shared analytical dataset: 50


In [133]:
# verify every manifest variable exists

missing_shared_cols = [
    col
    for col in shared_dataset_cols
    if col not in clean.columns
]

print(
    "Manifest variables missing from clean dataset:",
    missing_shared_cols
)

Manifest variables missing from clean dataset: []


In [134]:
assert len(missing_shared_cols) == 0

print(
    "All manifest variables are available."
)

All manifest variables are available.


### 9.2 Create the Shared Analytical Dataset

In [135]:
# subset `clean` using the manifest

shared_analytical = (
    clean[
        shared_dataset_cols
    ]
    .copy()
)

In [136]:
# inspect

print(
    "Shared analytical dataset shape:",
    shared_analytical.shape
)

Shared analytical dataset shape: (3144, 50)


In [137]:
shared_analytical.head()

,fips,state_abbr,county_name_places,places_target_year,routine_checkup_pct,primary_care_physicians_per_100k,nurse_practitioners_per_100k,physician_assistants_per_100k,hospitals_per_100k,hospital_beds_per_1000,...,mh_hpsa_max_score,dh_hpsa_max_score,popn_est_22,phys_nf_prim_care_pc_exc_rsdt_22,np_npi_22,pa_npi_22,hosp_22,hosp_beds_22,fedly_qualfd_hlth_ctr_22,rural_hlth_clincs_22
0,06045,CA,Mendocino,2022,68.6,80.193355,83.534745,32.300101,3.341390,1.314280,...,19.0,18.0,89783.0,72.0,75.0,29.0,3.0,118.0,14.0,7.0
1,02063,AK,Chugach,2022,62.2,145.475706,14.547571,29.095141,0.000000,0.000000,...,NaN,NaN,6874.0,10.0,1.0,2.0,0.0,0.0,NaN,NaN
2,04007,AZ,Gila,2022,68.8,50.072327,53.781388,38.945143,3.709061,1.131264,...,20.0,24.0,53922.0,27.0,29.0,21.0,2.0,61.0,2.0,6.0
3,05087,AR,Madison,2022,74.5,22.875443,51.469747,0.000000,0.000000,0.000000,...,15.0,18.0,17486.0,4.0,9.0,0.0,0.0,0.0,1.0,0.0
4,05067,AR,Jackson,2022,76.7,30.076997,102.261790,12.030799,6.015399,2.045236,...,18.0,11.0,16624.0,5.0,17.0,2.0,1.0,34.0,2.0,0.0


### 9.3 Structural Validation

The shared analytical dataset must preserve the structural guarantees established during the data-audit stage:

- 3,144 county/county-equivalent observations,
- one unique five-character FIPS code per county,
- no duplicate county records, and
- complete supervised outcome coverage.

In [138]:
print(
    "Rows:",
    len(shared_analytical)
)

print(
    "Columns:",
    shared_analytical.shape[1]
)

print(
    "Unique FIPS:",
    shared_analytical[
        "fips"
    ].nunique()
)

print(
    "Duplicate FIPS:",
    shared_analytical[
        "fips"
    ].duplicated().sum()
)

print(
    "Missing target:",
    shared_analytical[
        target_col
    ].isna().sum()
)

Rows: 3144
Columns: 50
Unique FIPS: 3144
Duplicate FIPS: 0
Missing target: 0


In [139]:
assert len(shared_analytical) == 3144

assert (
    shared_analytical["fips"]
    .nunique()
    == 3144
)

assert (
    shared_analytical["fips"]
    .duplicated()
    .sum()
    == 0
)

assert (
    shared_analytical[
        target_col
    ]
    .isna()
    .sum()
    == 0
)

assert (
    shared_analytical[
        "fips"
    ]
    .str.len()
    .eq(5)
    .all()
)

print(
    "Shared analytical dataset "
    "structural validation passed."
)

Shared analytical dataset structural validation passed.


### 9.4 Confirm Analytical Groups Are Represented

In [140]:
# final handoff check

analysis_group_validation = pd.DataFrame({
    "feature_group": [
        "Primary supervised",
        "Supervised sensitivity",
        "SVI supervised sensitivity",
        "Primary unsupervised",
        "Unsupervised sensitivity",
        "HPSA score sensitivity"
    ],

    "expected_n": [
        len(supervised_primary_candidates),
        len(supervised_sensitivity_candidates),
        len(supervised_svi_sensitivity),
        len(unsupervised_primary_candidates),
        len(unsupervised_sensitivity_candidates),
        len(unsupervised_hpsa_score_sensitivity)
    ],

    "available_n": [
        sum(
            col in shared_analytical.columns
            for col in supervised_primary_candidates
        ),

        sum(
            col in shared_analytical.columns
            for col in supervised_sensitivity_candidates
        ),

        sum(
            col in shared_analytical.columns
            for col in supervised_svi_sensitivity
        ),

        sum(
            col in shared_analytical.columns
            for col in unsupervised_primary_candidates
        ),

        sum(
            col in shared_analytical.columns
            for col in unsupervised_sensitivity_candidates
        ),

        sum(
            col in shared_analytical.columns
            for col in unsupervised_hpsa_score_sensitivity
        )
    ]
})

analysis_group_validation

,feature_group,expected_n,available_n
0,Primary supervised,10,10
1,Supervised sensitivity,5,5
2,SVI supervised sensitivity,16,16
3,Primary unsupervised,27,27
4,Unsupervised sensitivity,6,6
5,HPSA score sensitivity,3,3


In [141]:
assert (
    analysis_group_validation[
        "expected_n"
    ]
    ==
    analysis_group_validation[
        "available_n"
    ]
).all()

print(
    "All defined analytical feature groups "
    "are represented."
)

All defined analytical feature groups are represented.


### 9.5 Analysis Views (for convenience)

#### Note to teammates:

Here is how to pull the appropriate columns from the same shared file.

##### Primary supervised view 

In [142]:
supervised_primary_view = (
    shared_analytical[
        identifier_cols
        + [target_col]
        + supervised_primary_candidates
    ]
    .copy()
)

print(
    "Primary supervised view:",
    supervised_primary_view.shape
)

Primary supervised view: (3144, 15)


Contains:

```
identifiers
+ outcome
+ primary supervised predictors
```

##### Broader supervised sensitivity view

In [143]:
supervised_svi_view = (
    shared_analytical[
        identifier_cols
        + [target_col]
        + supervised_primary_candidates
        + supervised_svi_sensitivity
    ]
    .copy()
)

In [144]:
# avoid duplicate columns just in case:

supervised_svi_view = supervised_svi_view.loc[
    :,
    ~supervised_svi_view.columns.duplicated()
]

In [145]:
print(
    "Supervised + SVI sensitivity view:",
    supervised_svi_view.shape
)

Supervised + SVI sensitivity view: (3144, 31)


This corresponds to:

```
Primary healthcare-resource model
        +
SVI vulnerability indicators
```

##### Primary unsupervised view

For clustering, include identifiers for interpretation but not the outcome as a clustering feature:

In [146]:
unsupervised_primary_view = (
    shared_analytical[
        identifier_cols
        + unsupervised_primary_candidates
    ]
    .copy()
)

In [147]:
print(
    "Primary unsupervised view:",
    unsupervised_primary_view.shape
)

Primary unsupervised view: (3144, 31)


In [148]:
# to separately retain the outcome for later cluster interpretation:

cluster_outcome_reference = (
    shared_analytical[
        identifier_cols
        + [target_col]
    ]
    .copy()
)

---

### Shared Dataset Workflow

```text
                 master_county_analytical.csv
                           │
                           │
             same 3,144-county dataset
                           │
              ┌────────────┴────────────┐
              │                         │
              ▼                         ▼
     SUPERVISED ANALYSIS        UNSUPERVISED ANALYSIS
              │                         │
      feature manifest            feature manifest
              │                         │
              ▼                         ▼
  healthcare-resource +      healthcare-resource +
  Primary Care HPSA          PC / MH / Dental HPSA +
  predictors                 SVI indicators
              │                         │
              ▼                         ▼
     predict routine           identify county
     checkup prevalence        access-vulnerability
                               profiles
                                        │
                                        ▼
                              compare clusters with
                              routine-checkup outcome
                              AFTER clustering



Both workflows use the same underlying county records. Analytical feature groups determine which columns are used for each task.


### 9.6 Save the Shared Analytical Dataset

Saved under `data/processed/`.

In [149]:
analytical_path = (
    PROCESSED_DIR
    / "master_county_analytical.csv"
)

shared_analytical.to_csv(
    analytical_path,
    index=False
)

print(
    "Saved shared analytical dataset to:",
    analytical_path
)

Saved shared analytical dataset to: ../data/processed/master_county_analytical.csv


#### Save feature manifest

In [150]:
feature_manifest_path = (
    REFERENCE_DIR
    / "feature_manifest.csv"
)

feature_manifest.to_csv(
    feature_manifest_path,
    index=False
)

print(
    "Saved feature manifest to:",
    feature_manifest_path
)

Saved feature manifest to: ../data/reference/feature_manifest.csv


In [151]:
# save feature-group summary

feature_group_summary_path = (
    CLEANING_RESULTS_DIR
    / "feature_group_summary.csv"
)

feature_group_summary.to_csv(
    feature_group_summary_path,
    index=False
)

print("Saved feature-group summary to:", feature_group_summary_path)

Saved feature-group summary to: ../results/cleaning/feature_group_summary.csv


In [152]:
# save raw-to-engineered mapping:

raw_engineered_mapping.to_csv(
    CLEANING_RESULTS_DIR
    / "raw_to_engineered_mapping.csv",
    index=False
)

### 9.7 Reload Validation

In [153]:
shared_check = pd.read_csv(
    analytical_path,
    dtype={
        "fips": "string"
    },
    low_memory=False
)

In [154]:
print(
    "Reloaded shape:",
    shared_check.shape
)

print(
    "Unique FIPS:",
    shared_check[
        "fips"
    ].nunique()
)

print(
    "Duplicate FIPS:",
    shared_check[
        "fips"
    ].duplicated().sum()
)

print(
    "Missing target:",
    shared_check[
        target_col
    ].isna().sum()
)

Reloaded shape: (3144, 50)
Unique FIPS: 3144
Duplicate FIPS: 0
Missing target: 0


In [155]:
# validate

assert (
    shared_check.shape
    ==
    shared_analytical.shape
)

assert (
    shared_check[
        "fips"
    ]
    .str.len()
    .eq(5)
    .all()
)

assert (
    shared_check[
        "fips"
    ].nunique()
    == 3144
)

assert (
    shared_check[
        "fips"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    shared_check[
        target_col
    ]
    .isna()
    .sum()
    == 0
)

print(
    "Saved analytical dataset "
    "reload validation passed."
)

Saved analytical dataset reload validation passed.


### 9.8 Team Handoff

The shared analytical dataset is:

`data/processed/master_county_analytical.csv`

The corresponding feature definitions are stored in:

`data/reference/feature_manifest.csv`

All downstream analyses should begin from this common dataset rather than independently re-cleaning or re-merging the raw PLACES, AHRF, HPSA, or SVI files.

#### Supervised workflow

Use the feature manifest entries corresponding to:

- `supervised`
- `supervised_sensitivity`
- `supervised_svi_sensitivity`

The supervised modeling workflow remains responsible for model-specific:

- train/test or cross-validation splits,
- imputation,
- categorical encoding,
- scaling where required,
- feature selection,
- hyperparameter tuning,
- and model evaluation.

These operations should be fitted within the training/CV pipeline to prevent data leakage.

#### Unsupervised workflow

Use the feature manifest entries corresponding to:

- `unsupervised`
- `unsupervised_sensitivity`
- `unsupervised_hpsa_score_sensitivity`

The unsupervised workflow remains responsible for:

- final clustering-feature refinement,
- missing-data treatment,
- scaling,
- PCA,
- clustering,
- cluster-number selection,
- stability assessment,
- and cluster interpretation.

The PLACES routine-checkup outcome must not be used to construct clusters. It may be added after clustering for external descriptive comparison.

---

### Shared Analytical Dataset Result

The cleaning and feature-engineering workflow produced a common analytical dataset containing 3,144 U.S. counties and county equivalents.

The dataset includes 50 unique variables covering:

- county identifiers,
- the routine-checkup outcome,
- population-standardized healthcare-resource measures,
- geographic and access-context variables,
- Primary Care, Mental Health, and Dental Health HPSA measures,
- 16 underlying SVI indicators,
- sensitivity-analysis variables, and
- selected source variables retained for provenance.

No counties were removed during shared data preparation.

The dataset intentionally remains partially unprocessed from a model-specific perspective. Missing-value imputation, scaling, categorical encoding, PCA, and other model-dependent transformations have not been globally applied.

Instead, the feature manifest defines which variables are intended for each analytical workflow, while downstream preprocessing remains the responsibility of the corresponding supervised or unsupervised pipeline.

**Shared analytical dataset:**  
`data/processed/master_county_analytical.csv`

**Feature manifest:**  
`data/reference/feature_manifest.csv`

**Decision:** Both supervised and unsupervised workflows will use this common processed dataset as their starting point.

---

## 10. Reproducibility Summary and Team Handoff

The shared cleaning and feature-engineering workflow is complete.

The final shared analytical dataset is:

`data/processed/master_county_analytical.csv`

The corresponding feature definitions are stored in:

`data/reference/feature_manifest.csv`

### Reproducibility Summary

The processed dataset:

- contains 3,144 unique counties and county equivalents,
- preserves standardized five-character FIPS codes,
- contains no duplicate county observations,
- retains complete routine-checkup outcome data,
- includes validated healthcare-resource, HPSA, and SVI features, and
- preserves raw/reference variables needed for provenance and sensitivity analysis.

No global model-specific imputation, scaling, categorical encoding, PCA, or feature selection has been applied.